# Train `pyllm` (125M) on Kaggle GPU

Trains the from-scratch decoder-only Transformer at
[github.com/abhishekdubey-elets/python](https://github.com/abhishekdubey-elets/python)
end-to-end: **clone → build a Python corpus → clean/dedup/pack → train (fp16, KV-cache attention) → loss curve → sample.**

### Before you Run All
1. **Settings → Accelerator → GPU** (P100 or T4 — a single GPU is used).
2. *(Optional, for the better corpus)* **Settings → Internet → On**. With internet the notebook streams a slice of the
   `codeparrot/codeparrot-clean` GitHub-Python corpus. Without it, it falls back to the Kaggle image's own Python
   (stdlib + site-packages) — fully offline, still a real training run.

This is the **quick run** profile: 125M params, ~1000 steps, roughly **20–45 min** on a T4/P100. All knobs are in the
**Config** cell. Checkpoints, the loss curve, and packed data are written to `/kaggle/working/` so they persist as
notebook output you can download.


## 1 · Environment & GPU check

In [ ]:
import sys, torch
print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__)
cuda = torch.cuda.is_available()
print("CUDA   :", cuda)
if cuda:
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   : %.1f GB" % (props.total_memory / 1e9))
    print("bf16   :", torch.cuda.is_bf16_supported(), "(pre-Ampere GPUs -> we train in fp16)")

    # torch.cuda.is_available() only reports a driver, not that this torch build ships
    # kernels for this chip. A P100 (sm_60) on a torch built for sm_70+ passes every
    # check above, then dies on the first real matmul mid-training. Fail fast instead.
    arches = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
    cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
    print("arch   :", cap, "| torch builds for:", " ".join(arches))
    if cap not in arches:
        print()
        print("INCOMPATIBLE GPU")
        print("  " + props.name + " is " + cap + ", but torch " + torch.__version__ + " has no " + cap + " kernels.")
        print("  Training would crash: no kernel image is available for execution on the device.")
        print("  Fix: Notebook Settings > Accelerator > GPU T4 x2  (T4 is sm_75, which is supported).")
        raise RuntimeError("Incompatible GPU: " + props.name + " (" + cap + ") -- switch the accelerator to GPU T4 x2.")
    print("OK     : this torch has kernels for", cap)
else:
    print("No GPU detected -> enable it in Settings > Accelerator. The notebook will still run on CPU (slow).")

## 2 · Clone the repo

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/abhishekdubey-elets/python.git"
REPO_DIR = "/kaggle/working/python-llm"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# The package lives under src/ (src-layout); importing it just needs src on the path.
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# Light deps only — Kaggle already ships torch, numpy, matplotlib.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.15", "pyyaml>=6.0"], check=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("repo ready at", os.getcwd())

## 3 · Config

Every knob for the run. Defaults = the **125M quick run**. Raise `MAX_STEPS` (and optionally `MICRO_BATCH`) for a
longer/heavier run; drop them for a faster smoke test.

In [ ]:
# --- data ---------------------------------------------------------------
USE_HF       = True     # stream codeparrot if Internet is ON; else auto-fallback to local Python
MAX_DOCS     = 25000    # clean documents to keep (governs corpus/token count)
NEAR_DEDUP   = False    # MinHash near-dup pass — slower; exact dedup always runs
VAL_FRACTION = 0.05

# --- model / paths ------------------------------------------------------
MODEL_CFG = "configs/model_125m.yaml"
TOKENIZER = "tokenizer/pyllm.json"
DATA_DIR  = "/kaggle/working/data/packed"
CKPT_DIR  = "/kaggle/working/checkpoints"

# --- training (quick-run profile) --------------------------------------
MAX_STEPS    = 1000
WARMUP_STEPS = 100
MICRO_BATCH  = 8        # sequences/forward. Fits 16 GB comfortably; try 12–16 if VRAM allows.
GRAD_ACCUM   = 4        # effective batch = MICRO_BATCH * GRAD_ACCUM (memory-free to raise)
LR, MIN_LR   = 6e-4, 6e-5
EVAL_INTERVAL, EVAL_ITERS, LOG_INTERVAL = 250, 40, 10
SEED = 1337
print("effective batch:", MICRO_BATCH * GRAD_ACCUM, "sequences/step")

## 4 · Build the Python corpus

The repo ships the pipeline and tokenizer but **no data**. We assemble a list of Python source strings — from
`codeparrot/codeparrot-clean` if the internet is reachable, otherwise from the Kaggle image's own Python installation.
A fast socket probe decides, so there are no long hangs when internet is off.

In [ ]:
import socket, sys, subprocess

def internet_up(host="huggingface.co", port=443, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.create_connection((host, port)).close()
        return True
    except OSError:
        return False

def collect_hf(target):
    """Stream a slice of codeparrot-clean; over-collect since cleaning drops some."""
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
        from datasets import load_dataset
    ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)
    texts, scan_cap = [], target * 6
    for i, ex in enumerate(ds):
        t = ex.get("content") or ex.get("text") or ""
        if t:
            texts.append(t)
        if len(texts) >= target * 2 or i >= scan_cap:
            break
    return texts

def collect_local(target):
    """Walk the Kaggle image's stdlib + site-packages for .py files."""
    import sysconfig
    roots, seen = [], set()
    for key in ("stdlib", "purelib", "platlib"):
        r = sysconfig.get_paths().get(key)
        if r and r not in seen and os.path.isdir(r):
            roots.append(r); seen.add(r)
    files = []
    for root in roots:
        for dp, _dn, fns in os.walk(root):
            for name in fns:
                if name.endswith(".py"):
                    files.append(os.path.join(dp, name))
    files.sort()
    texts = []
    for fp in files:
        try:
            sz = os.path.getsize(fp)
            if sz < 80 or sz > 200_000:      # skip trivial + giant/generated files
                continue
            with open(fp, encoding="utf-8", errors="ignore") as f:
                texts.append(f.read())
        except OSError:
            continue
        if len(texts) >= target * 2:
            break
    return texts

use_hf = USE_HF and internet_up()
raw_texts, source = [], None
if use_hf:
    try:
        raw_texts = collect_hf(MAX_DOCS); source = "huggingface:codeparrot-clean"
    except Exception as e:
        print("HF path failed -> local fallback. Reason:", repr(e)); raw_texts = []
if not raw_texts:
    raw_texts = collect_local(MAX_DOCS); source = "local:kaggle-python"

print(f"collected {len(raw_texts):,} raw documents from {source}")

## 5 · Clean → dedup → tokenize → pack

Uses the repo's own pipeline: `clean_document` (normalize + quality gate + `ast.parse` validity), exact content-hash
dedup, optional MinHash near-dedup, then tokenize (byte-level BPE, EOS-separated) and pack into `uint16` shards.
*Cleaning parses every file, so this can take a couple of minutes.*

In [ ]:
from pyllm.data import clean_document, dedup_exact, dedup_near, pack_dataset
from pyllm.tokenizer import PyTokenizer

cleaned = [d for d in (clean_document(t) for t in raw_texts) if d is not None]
print("after clean/validate:", f"{len(cleaned):,}")

docs = [cleaned[i] for i in dedup_exact(cleaned)]
print("after exact dedup   :", f"{len(docs):,}")

if NEAR_DEDUP:
    docs = [docs[i] for i in dedup_near(docs)]
    print("after near dedup    :", f"{len(docs):,}")

docs = docs[:MAX_DOCS]
tok = PyTokenizer.load(TOKENIZER)
stats = pack_dataset(docs, tok, DATA_DIR, val_fraction=VAL_FRACTION, seed=SEED)
print("packed ->", DATA_DIR)
for k, v in stats.items():
    print(f"  {k:12s}: {v}")

## 6 · Build model + training config

In [ ]:
import torch
from pyllm.config import ModelConfig, TrainConfig
from pyllm.model import PyLLM
from pyllm.data import load_bin

model_cfg = ModelConfig.from_yaml(MODEL_CFG)

# Keep tokenizer and model vocab in lockstep (train.py would otherwise reject a mismatch).
if tok.vocab_size != model_cfg.vocab_size:
    d = model_cfg.to_dict(); d["vocab_size"] = tok.vocab_size
    model_cfg = ModelConfig.from_dict(d)
    print("adjusted model vocab_size ->", tok.vocab_size)

cuda = torch.cuda.is_available()
train_cfg = TrainConfig(
    lr=LR, min_lr=MIN_LR, weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0,
    warmup_steps=WARMUP_STEPS, max_steps=MAX_STEPS,
    micro_batch_size=MICRO_BATCH if cuda else 2,
    grad_accum_steps=GRAD_ACCUM,
    device="cuda" if cuda else "cpu",
    dtype="float16" if cuda else "float32",   # P100/T4 lack bf16; fp16 + GradScaler (handled by Trainer)
    compile=False, grad_checkpointing=False, seed=SEED,
    eval_interval=EVAL_INTERVAL, eval_iters=EVAL_ITERS, log_interval=LOG_INTERVAL,
    checkpoint_dir=CKPT_DIR,
)

train_data = load_bin(f"{DATA_DIR}/train.bin")
val_data   = load_bin(f"{DATA_DIR}/val.bin")

model = PyLLM(model_cfg)
tps = train_cfg.tokens_per_step(model_cfg.seq_len)
print(f"model         : {model.num_params()/1e6:.1f}M params on {train_cfg.device} ({train_cfg.dtype})")
print(f"train tokens  : {len(train_data):,}  |  val tokens: {len(val_data):,}")
print(f"tokens/step   : {tps:,}  ->  {MAX_STEPS} steps see ~{tps*MAX_STEPS/1e6:.0f}M tokens "
      f"(~{tps*MAX_STEPS/max(len(train_data),1):.1f} epochs)")

## 7 · Train

nanoGPT-style loop: cosine LR with warmup, gradient accumulation, grad clipping, fp16 mixed precision, periodic eval,
and a checkpoint written to `/kaggle/working/checkpoints/ckpt.pt` at every eval.

In [ ]:
import time
from pyllm.training import Trainer

trainer = Trainer(model, model_cfg, train_cfg, train_data, val_data, save_checkpoints=True)
t0 = time.time()
history = trainer.train()
print(f"\ndone in {(time.time()-t0)/60:.1f} min -> checkpoint at {CKPT_DIR}/ckpt.pt")

## 8 · Loss curve

In [ ]:
import matplotlib.pyplot as plt

steps = history["step"]
plt.figure(figsize=(7, 4))
plt.plot(steps, history["train_loss"], marker="o", label="train")
if any(v is not None for v in history["val_loss"]):
    plt.plot(steps, history["val_loss"], marker="s", label="val")
plt.xlabel("step"); plt.ylabel("cross-entropy loss")
plt.title("pyllm 125M — Kaggle quick run"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 9 · Sample from the trained model

Reloads the checkpoint and generates with the KV-cached sampler. Generation runs on **CPU** (128 tokens is a few
seconds) to sidestep CUDA RNG-device quirks. After only ~1000 steps expect *Python-shaped* output — indentation,
`def`/`return`, plausible identifiers — not correct programs.

In [ ]:
import torch
from pyllm.model import PyLLM
from pyllm.inference import generate
from pyllm.training import load_checkpoint, model_config_from_checkpoint

ckpt = load_checkpoint(f"{CKPT_DIR}/ckpt.pt", map_location="cpu")
gen_model = PyLLM(model_config_from_checkpoint(ckpt))
gen_model.load_state_dict(ckpt["model"]); gen_model.eval()

for prompt in ["def fibonacci(n):", "import numpy as np\n", "class Stack:"]:
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long)
    out = generate(gen_model, ids, max_new_tokens=96, temperature=0.8, top_p=0.95,
                   eos_id=tok.eos_id, generator=torch.Generator().manual_seed(0))
    print("=" * 70)
    print(tok.decode(out[0].tolist()))

## Outputs & next steps

Everything under `/kaggle/working/` is downloadable from the notebook's **Output** tab (or **Save Version** to persist):

| File | What |
|------|------|
| `checkpoints/ckpt.pt` | model + optimizer + configs (self-describing; resumable) |
| `loss_curve.png` | the training curve above |
| `data/packed/{train,val}.bin`, `meta.json` | the tokenized corpus |

**To train harder** (bring the loss down): flip **Internet On** + keep `USE_HF=True` for the codeparrot corpus, then in
the Config cell raise `MAX_STEPS` (e.g. 5000–20000) and, if VRAM allows, `MICRO_BATCH` to 12–16. A Kaggle session lasts
~12 h and free GPU quota is ~30 h/week, so save a version before long runs.

**300M model**: switch `MODEL_CFG = "configs/model_300m.yaml"`, drop `MICRO_BATCH` to 4, and set
`grad_checkpointing=True` in the `TrainConfig` to fit 16 GB.
